In [ ]:
# PyTorch functions/methods helpers

#6.1.1
assert torch.allclose(Y.mean(), torch.tensor(0.0)) # Pass, as Y.mean() should be or is extremely close to 0

#6.1.2
assert torch.equal(Y, torch.full((2, 3), 2.0)) # Y is equal to a tensor of shape (2, 3) that only contains 2.0 as elements

#6.1.3
for name, module in net.named_children(): # net.named_children() gives each immediate layer along with its name
    current = module(current) # Taking the current data and passing it through the current layer e.g. module.forward(current)
    shape_trace.append((name, type(module).__name__, shape(current)))

#6.1.4 (custom function made by codex)
print("parameter scalars", count_scalars(block.parameters())) # count the number of elements/params in a collection of tensors

* Chapter 5 treated networks mostly as computations: tensors -> predictions -> loss -> gradients -> param update.

* Chapter 6 starts the software step: package those computations to be inspected, reused, nested, saved, moved to a GPU, and trained without losing track of the model's state.

* A PyTorch module is the boundary where a mathematical idea becomes a reliable software object.

# How to use this notebook

* Run the notebook from top to bottom.

* Every code block is designed to be cloud-runnable and self-contained inside this notebook.

* The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

# You are done when you can

- explain why deep learning code needs modules instead of loose tensor functions everywhere
- explain what `forward` owns and what `Module.__call__` adds around it
- trace tensor shapes through a module stack
- read a module as a representation pipeline, not just a list of APIs
- distinguish simple chains from custom forward logic
- debug a shape mismatch inside a module stack

In [ ]:
import math
from pathlib import Path
import tempfile

import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
  return tuple(x.shape)

def count_scalars(parameters):
  return sum(p.numel() for p in parameters)

# 6.1.0 The Problem This Notebook Solves

Before this point, it is possible to understand neural networks as loose pieces:

- a weight tensor
- a bias tensor
- a forward computation
- a loss
- a backward pass
- an optimizer step


That is enough for tiny examples, but it does not scale. Once a model has many layers, you need a way to answer basic engineering questions without manually remembering everything:

- Which tensors are part of the model?
- Which pieces are trainable?
- Which computation runs first, second, and third?
- What shape does each layer expect?
- How do we move the whole model to another device?
- How do we save and reload the model's learned state?
- How do we reuse the same block in a larger architecture?

`nn.Module` exists to solve that organization problem. It is not just decoration around a function. It gives PyTorch a standard way to treat model components as objects with structure, state, and behavior.


The mental model for this chapter is:

```text
__init__ defines what the component owns
forward defines what the component does to inputs
module(X) runs the component through PyTorch's normal call path
```

That separation matters.

* If a tensor is created inside `forward` and not stored on `self`, it is usually temporary computation.

* If a layer, parameter, or buffer is assigned to `self` in `__init__`, PyTorch can discover it later.

* Chapter 6.2 will build on this by asking exactly which parameters a module owns.

* For now, the goal is to understand the module as the basic boundary of model code.

# 6.1.1 A Module Is a Callable Object With a Forward Contract

Start with the smallest possible module: one that has computation but no trainable parameters. This is important because it separates two ideas that beginners often merge together:

- A module is a reusable computation boundary.
- A parameter is a learned tensor.

A module can have parameters, but it does not need them. A centering layer is still a meaningful model component because it changes the representation passed to later layers. It receives a tensor, computes the mean, subtracts that mean, and returns another tensor with the same shape.

The theoretical meaning is simple: some layers transform the coordinate system of the data without learning anything. They can normalize, reshape, clip, center, mask, or route values.

These operations affect what later trainable layers see, so they belong in the model pipeline even when they have no weights.

Before running the cell, predict:

- The output shape should match the input shape.
- The output mean should be zero or extremely close to zero.
- `layer(X)` should call `forward(X)` indirectly.

In normal PyTorch code, call `layer(X)`, not `layer.forward(X)`. The direct `forward` call skips PyTorch's normal module call path. That path is where hooks, wrappers, instrumentation, and other framework features can attach.


In [ ]:
class CenteredLayer(nn.Module):

  def __init__(self):
    super().__init__()

  def forward(self, X):
    return X - X.mean()

X = torch.tensor([1.0, 2.0, 3.0, 4.0]) # Shape of (4, )
layer = CenteredLayer()
Y = layer(X) # Same shape to X of (4, )

print("input:", X) # [1.0, 2.0, 3.0, 4.0]
print("output:", Y) # [1.0-2.5, 2.0-2.5, 3.0-2.5, 4.0-2.5] or [-1.5, -0.5, 0.5, 1.5]
print("output mean:", Y.mean()) # [0]

assert shape(Y) == shape(X) # Pass
assert torch.allclose(Y.mean(), torch.tensor(0.0)) # Pass, as Y.mean() should be or is extremely close to 0

input: tensor([1., 2., 3., 4.])
output: tensor([-1.5000, -0.5000,  0.5000,  1.5000])
output mean: tensor(0.)


# 6.1.2 The Call Path Preserves State on the Module

A neural network layer is not only a pure mathematical expression. In software, it is also an object that can remember things.

That object state is what lets larger systems inspect the model, save it, move it between devices, and configure training.

There are several kinds of state a module might own:

- ordinary Python attributes, such as a debug flag or the last input shape
- submodules, such as a linear layer inside a larger block
- parameters, which are tensors learned by gradient descent
- buffers, which are persistent tensors that are not usually learned by gradients

This cell uses ordinary Python state only: `last_input_shape`. That makes the idea visible without introducing parameter registration yet.

The discipline is:

- `__init__` should create the structure and long-lived state.
- `forward` should describe the computation for one call.
- temporary tensors created inside `forward` are part of the current computation, not permanent model ownership.

The graceful handoff is that this simple logger prepares you for parameterized modules.
* A `Linear` layer is not conceptually different in shape: it is also an object that owns state and defines a forward computation.
* The difference is that some of its state is trainable.

In [ ]:
class ShapeLogger(nn.Module):

    def __init__(self):
        super().__init__()
        self.last_input_shape = None

    def forward(self, X):
        self.last_input_shape = shape(X)
        return X * 2 # Every element in X is multiplied by 2

logger = ShapeLogger()
X = torch.ones(2, 3) # Create a tensor with the shape of (2, 3)
Y = logger(X)

print("last input shape:", logger.last_input_shape)
print("output shape:", shape(Y))

assert logger.last_input_shape == (2, 3)
assert torch.equal(Y, torch.full((2, 3), 2.0)) # Y is equal to a tensor of shape (2, 3) that only contains 2.0 as elements

last input shape: (2, 3)
output shape: (2, 3)


# 6.1.3 Sequential Modules Are Shape Pipelines

Most beginner networks are representation pipelines. Each layer takes the current representation and produces the next one.

For an MLP, this usually means:

```text
raw features -> hidden representation -> activated hidden representation -> output scores
```

The word "representation" matters. A hidden layer is not just a tensor with a different size. It is the model's current internal description of the input.

* The first `Linear(4, 5)` layer maps each example from 4 **input features** into 5 **hidden features**.

* `ReLU` keeps the same shape but changes the values by suppressing negative activations.

* The final `Linear(5, 2)` maps the 5 **hidden features** into 2 **output values**.


`nn.Sequential` is appropriate when this story is a straight chain. The key constraint becomes a mechanical shape handshake:

```text
each layer's output feature count must match the next layer's expected input feature count
```

The shape trace in the next cell is not busywork. It is how you verify that the representation pipeline you intended is actually built.

In [ ]:
net = nn.Sequential(
    nn.Linear(4, 5),
    nn.ReLU(),
    nn.Linear(5, 2),
)

X = torch.randn(3, 4)
shape_trace = []
current = X

for name, module in net.named_children(): # net.named_children() gives each immediate layer along with its name
    current = module(current) # Taking the current data and passing it through the current layer e.g. module.forward(current)
    shape_trace.append((name, type(module).__name__, shape(current)))

for row in shape_trace:
    print(row)

assert shape_trace == [
    ("0", "Linear", (3, 5)),
    ("1", "ReLU", (3, 5)),
    ("2", "Linear", (3, 2)),
]

('0', 'Linear', (3, 5))
('1', 'ReLU', (3, 5))
('2', 'Linear', (3, 2))


# 6.1.4 Custom Forward Logic Is for Branches, Reuse, and Control Flow

`nn.Sequential` is a chain. Real architectures are often not pure chains.

They may reuse a block, branch into multiple paths, add a skip connection, or make a decision based on the input.

This is where custom modules become more than syntax. They let you express the actual structure of the model.

The example below is a residual MLP block. Its idea is:

```text
keep the original representation
compute an update to that representation
add the update back to the original
```

The theoretical meaning is that the block does not have to reinvent the whole representation from scratch. It can learn a correction.

This is one of the central ideas behind residual networks: if the best thing for a layer to do is "mostly preserve what came in, with a learned adjustment", the architecture makes that easy to express.

The important shape rule is strict: **if you add two tensors, they must be compatible shapes**. Here the block must return the same shape as its input because `X + update` is the core operation.

Before running the cell, predict:

- Input shape: batch size 2, feature width 4.
- Output shape: still batch size 2, feature width 4.
- Parameter count: two `Linear(4, 4)` layers, each with a 4 by 4 weight and a 4-value bias.

In [ ]:
class ResidualMLPBlock(nn.Module):

    def __init__(self, width):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(width, width), # shape of X (2, 4) @ Layer 1 weight.T shape of (4, 4) = shape of (2, 4)
            nn.ReLU(),
            nn.Linear(width, width), # shape from Layer 1 (2, 4) @ Final Layer weight.T shape of (4, 4) = shape of (2, 4)
        )

    def forward(self, X):
        update = self.block(X)
        return X + update

block = ResidualMLPBlock(width=4)
X = torch.randn(2, 4)
Y = block(X)

print("input shape:", shape(X)) # (2, 4)
print("output shape:", shape(Y)) # (2, 4)
print("parameter scalars", count_scalars(block.parameters())) # 2 weights with shape of (4, 4), and 2 bias with shape of (4, ), so 4*4*2 + 4*2 = 16*2 + 8 = 40

assert shape(Y) == shape(X)
assert count_scalars(block.parameters()) == 4 * 4 + 4 + 4 * 4 + 4

input shape: (2, 4)
output shape: (2, 4)
parameter scalars 40


# 6.1.5 Break It Deliberately: Bad Shape Handoff

Shape errors are not random PyTorch annoyances. They are failed contracts between representation stages.

In the bad network below, the first linear layer maps 4 features to 5 hidden features. After `ReLU`, the tensor still has 5 features.

The final linear layer incorrectly expects 4 features. The code is trying to multiply tensors whose inner dimensions do not agree.

The theory-level mistake is: the architecture story is inconsistent.

* One block says "I produce a 5-feature representation";
* the next block says "I consume a 4-feature representation."
* The runtime error is just PyTorch discovering that contradiction.

This cell catches the error so the notebook keeps running. Read the first line of the error and connect it back to the broken handoff.

In [ ]:
bad_net = nn.Sequential(
    nn.Linear(4, 5), # shape of (3, 4) @ Layer 1 weight.T shape of (4, 5) = shape of (3, 5)
    nn.ReLU(),
    nn.Linear(4, 2), # shape from Layer 1 (3, 5) @ Final Layer weight.T shape of (4, 2) ???
)

try:
    bad_net(torch.randn(3, 4))
except RuntimeError as err:
    print(type(err).__name__)
    print(str(err).splitlines()[0]) # mat means matrix multiplication "@"
else:
    raise AssertionError("The bad network should have failed.")

RuntimeError
mat1 and mat2 shapes cannot be multiplied (3x5 and 4x2)


# 6.1 Checkpoint

Answer these before moving on.

You do not need a separate notes file for chapters; short answers in markdown cells or in your own study notes are enough.

1. What problem does `nn.Module` solve once networks become larger than a few loose tensors?
> `nn.Module` organizes layers and parameters into a reusable object that PyTorch can track, inspect, and manage

2. What belongs in `__init__`, and what belongs in `forward`?
> `__init__` defines/creates the layers and parameters the module has
> `forward` defines how data flows through those layers

3. Why should normal code call `module(X)` instead of `module.forward(X)`?
> `module(X)` lets PyTorch's `nn.Module` machinery run the forward pass properly, including things such as hooks and other framework behavior
> Whereas `module.forward(X)` directly calls the `forward` method, bypassing the normal `nn.Module` call machinery

4. Why can a layer with no parameters still be a legitimate module?
> Because it can still perform operations on the input

5. In the sequential example, why does the second `Linear` layer need `in_features=5`?
> Because the second layer's input features should match the first layer's result, or the first layer's output features

6. Why is a hidden layer better understood as a representation than as just a matrix output?
> A hidden layer transforms the original input into a new representation that can make useful patterns easier for later layers to work with

7. What shape must a residual block return if it adds its output back to its input?
> The same shape as the user input

8. When is `nn.Sequential` too limited?
> `nn.Sequential` is too limited when the network needs branching, skip/residual connections, multiple inputs or outputs, or other custom control flow that isn't just one layer after another